# 🧠 IA-NAHA — Notebook Machine Learning
### Prédiction des besoins de récupération (sommeil) et recommandations nutritionnelles via CIQUAL

---

**Problématique :** Comment l'IA peut-elle utiliser les données de vie quotidienne et les constantes biométriques pour prédire les besoins de récupération (sommeil) et recommander des ajustements nutritionnels automatisés via la base CIQUAL ?

**Tables utilisées :**
- `activite_globale.csv` → Activité quotidienne + sommeil (données principales)
- `sommeil_logs.csv` → Qualité du sommeil et troubles
- `gym_logs.csv` → Métabolisme de base et dépense calorique
- `ciqual_nutrition.csv` → Base alimentaire pour les recommandations
- `compendium_sports.csv` → Valeurs MET par activité sportive
- `profil_utilisateur.csv` → Profils démographiques

## 0️⃣ Installation des bibliothèques (à exécuter une seule fois sur Colab)

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn -q
print('✅ Bibliothèques installées')

## 1️⃣ Chargement des données

**Important :** Uploadez vos 6 fichiers CSV dans Colab via le panneau 📁 à gauche avant d'exécuter cette cellule.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# -------------------------------------------------------
# CHARGEMENT DES 6 TABLES
# Note : le fichier s'appelle activite_globale.csv mais on
# nomme la variable df_activite_globale pour coller
# au nom de ta table SQL dans MAMP
# -------------------------------------------------------

df_activite_globale = pd.read_csv('activite_globale.csv', sep=',')       
df_sommeil          = pd.read_csv('sommeil_logs.csv', sep=';')         # ta table sommeil_logs
df_gym              = pd.read_csv('gym_logs.csv', sep=';')             # ta table gym_logs
df_ciqual           = pd.read_csv('ciqual_nutrition.csv', sep=';', encoding='latin1')   # ta table ciqual_nutrition
df_compendium       = pd.read_csv('compendium_sports.csv', sep=';', encoding='latin1')  # ta table compendium_sports
df_profil           = pd.read_csv('profil_utilisateur.csv', sep=';')   # ta table profil_utilisateur

print('📦 Tables chargées avec succès !')
print(f'  → activite_globale   : {df_activite_globale.shape[0]} lignes, {df_activite_globale.shape[1]} colonnes')
print(f'  → sommeil_logs       : {df_sommeil.shape[0]} lignes, {df_sommeil.shape[1]} colonnes')
print(f'  → gym_logs           : {df_gym.shape[0]} lignes, {df_gym.shape[1]} colonnes')
print(f'  → ciqual_nutrition   : {df_ciqual.shape[0]} lignes, {df_ciqual.shape[1]} colonnes')
print(f'  → compendium_sports  : {df_compendium.shape[0]} lignes, {df_compendium.shape[1]} colonnes')
print(f'  → profil_utilisateur : {df_profil.shape[0]} lignes, {df_profil.shape[1]} colonnes')

## 2️⃣ Exploration rapide des données

In [ ]:
print('=== Aperçu de la table activite_globale ===')
display(df_activite_globale.head(3))
print('\n=== Valeurs manquantes ===')
manquants = df_activite_globale.isnull().sum()
print(manquants[manquants > 0])

print('\n=== Statistiques descriptives (variables clés) ===')
cols_cles = ['sleep_hours', 'stress_level', 'calories_burned', 'avg_heart_rate', 'bmi', 'daily_steps']
display(df_activite_globale[cols_cles].describe().round(2))

## 3️⃣ Préparation des données (Nettoyage + Ingénierie de variables)

On crée de nouvelles variables qui aident l'IA à mieux comprendre l'effort physique :

| Nouvelle variable | Calcul | Pourquoi ?
|---|---|---|
| `Charge_Cardiaque` | BPM moyen − BPM repos | Mesure l'intensité réelle de l'effort sur le cœur |
| `Charge_Activite` | Durée (min) × Calories | Combine la durée ET l'effort total dépensé |
| `Ratio_Pas_Calories` | Pas quotidiens / Calories | Distingue activité de fond vs sport intense |
| `IMC_Stress` | IMC × Stress | Interaction entre état corporel et mental |

In [ ]:
from sklearn.preprocessing import LabelEncoder

# ---- Copie de travail (on ne touche pas à df_activite_globale original) ----
df = df_activite_globale.copy()

# ---- Nettoyage ----
df = df.dropna(subset=['sleep_hours', 'stress_level', 'avg_heart_rate', 'resting_heart_rate'])
print(f'Lignes après nettoyage : {len(df)}')

# ---- Ingénierie de variables (Feature Engineering) ----
# On crée des variables qui aident l'IA à mieux comprendre l'effort physique

df['Charge_Cardiaque']   = df['avg_heart_rate'] - df['resting_heart_rate']
# Explication : écart entre BPM effort et BPM repos = intensité réelle sur le cœur

df['Charge_Activite']    = df['duration_minutes'] * df['calories_burned']
# Explication : durée × calories = effort total (30min intense ≠ 30min léger)

df['Ratio_Pas_Calories'] = df['daily_steps'] / (df['calories_burned'] + 1)
# Explication : quelqu'un qui marche toute la journée vs quelqu'un qui fait 1 séance intense

df['IMC_Stress']         = df['bmi'] * df['stress_level']
# Explication : interaction entre état corporel et état mental

# ---- Encodage texte → chiffres (l'IA ne comprend pas les mots) ----
le = LabelEncoder()
df['Genre_N']     = le.fit_transform(df['gender'].fillna('Unknown'))
df['Sport_N']     = le.fit_transform(df['activity_type'].fillna('Unknown'))
df['Intensite_N'] = le.fit_transform(df['intensity'].fillna('Unknown'))

print('✅ Variables créées !')
print(df[['Charge_Cardiaque','Charge_Activite','Ratio_Pas_Calories','IMC_Stress']].describe().round(2))

## 4️⃣ MODÈLE 1 — Prédire la durée du sommeil (Régression)

**Objectif :** À partir des données d'activité quotidienne, prédire combien d'heures l'utilisateur devrait dormir.

**Variable cible (y) :** `sleep_hours`  
**Variables explicatives (X) :** stress, activité cardiaque, IMC, calories, sport...

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

# ---- Variables explicatives choisies ----
features_modele1 = [
    'age', 'bmi', 'Genre_N', 'Sport_N', 'Intensite_N',
    'stress_level', 'calories_burned', 'daily_steps',
    'Charge_Cardiaque', 'Charge_Activite', 'Ratio_Pas_Calories', 'IMC_Stress',
    'blood_pressure_systolic', 'blood_pressure_diastolic',
    'hydration_level', 'duration_minutes'
]

X1 = df[features_modele1].fillna(df[features_modele1].median())
y1 = df['sleep_hours']

# ---- Séparation entraînement / test (80% / 20%) ----
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)
print(f'Entraînement : {len(X1_train)} exemples | Test : {len(X1_test)} exemples')

# ---- Comparaison de 3 algorithmes ----
modeles = {
    'Random Forest'       : RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
    'Gradient Boosting'   : GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
    'XGBoost'             : XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, verbosity=0)
}

resultats = {}
for nom, modele in modeles.items():
    modele.fit(X1_train, y1_train)
    preds = modele.predict(X1_test)
    mae   = mean_absolute_error(y1_test, preds)
    r2    = r2_score(y1_test, preds)
    cv    = cross_val_score(modele, X1, y1, cv=5, scoring='r2').mean()
    resultats[nom] = {'MAE': round(mae, 3), 'R²': round(r2, 3), 'R² CV (5-fold)': round(cv, 3)}
    print(f'  {nom:<25} | MAE={mae:.2f}h | R²={r2:.2f} | R² Cross-Val={cv:.2f}')

df_resultats = pd.DataFrame(resultats).T
print('\n=== Tableau comparatif des modèles ===')
display(df_resultats)

# On garde le meilleur
meilleur_nom = df_resultats['R²'].astype(float).idxmax()
modele_duree = modeles[meilleur_nom]
print(f'\n🏆 Meilleur modèle retenu : {meilleur_nom}')

### 4.1 Importance des variables — Quels facteurs impactent le plus le sommeil ?

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Récupération des importances
importances = pd.Series(modele_duree.feature_importances_, index=features_modele1)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(9, 6))
colors = ['#e74c3c' if v == importances.max() else '#3498db' for v in importances.values]
importances.plot(kind='barh', color=colors)
plt.title('Importance des variables — Prédiction durée du sommeil', fontsize=14, fontweight='bold')
plt.xlabel('Importance relative')
plt.tight_layout()
plt.show()

top3 = importances.sort_values(ascending=False).head(3)
print('🔍 Top 3 facteurs influençant le sommeil :')
for var, val in top3.items():
    print(f'   → {var} : {val*100:.1f}%')

## 5️⃣ MODÈLE 2 — Prédire la qualité du sommeil (Classification)

**Objectif :** Utiliser `sommeil_logs` pour entraîner un modèle qui prédit la qualité de sommeil (note de 1 à 10), puis appliquer ce modèle sur les données d'activité.

**Pourquoi deux tables ?** `activite_globale` contient les heures dormies mais pas la qualité. `sommeil_logs` contient la qualité. On va apprendre à prédire la qualité avec les colonnes communes (`age`, `stress_level`, `daily_steps`, `heart_rate`).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# ---- Préparation de sommeil_logs ----
df_s = df_sommeil.copy()
df_s['heart_rate']   = pd.to_numeric(df_s['heart_rate'], errors='coerce')
df_s['daily_steps']  = pd.to_numeric(df_s['daily_steps'], errors='coerce')
df_s['stress_level'] = pd.to_numeric(df_s['stress_level'], errors='coerce')
df_s = df_s.dropna(subset=['quality_of_sleep', 'stress_level', 'heart_rate', 'daily_steps'])

# Colonnes communes entre les deux tables
features_pont = ['age', 'stress_level', 'daily_steps', 'heart_rate', 'physical_activity_level']
X2 = df_s[features_pont]
y2 = df_s['quality_of_sleep'].astype(int)

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

modele_qualite = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
modele_qualite.fit(X2_train, y2_train)

y2_pred = modele_qualite.predict(X2_test)
acc = accuracy_score(y2_test, y2_pred)
cv2 = cross_val_score(modele_qualite, X2, y2, cv=5, scoring='accuracy').mean()

print(f'✅ Modèle Qualité entraîné sur {len(X2_train)} observations')
print(f'   Précision (Accuracy) : {acc:.2%}')
print(f'   Précision Cross-Val (5-fold) : {cv2:.2%}')
print()
print('=== Rapport de classification ===')
print(classification_report(y2_test, y2_pred))

In [ ]:
# ---- Application du Modèle 2 sur activite_globale ----
# On crée les colonnes communes pour faire le pont entre les deux tables
df_pred = df[['age', 'stress_level', 'daily_steps']].copy()
df_pred['heart_rate']              = df['avg_heart_rate']
df_pred['physical_activity_level'] = (df['duration_minutes'] / 60 * 10).clip(0, 100).astype(int)

df['Qualite_Predite'] = modele_qualite.predict(
    df_pred[features_pont].fillna(df_pred[features_pont].median())
)

print('✅ Qualité de sommeil prédite et ajoutée à la table activite_globale')
print(df[['participant_id', 'sleep_hours', 'stress_level', 'Qualite_Predite']].head(8).to_string(index=False))

## 6️⃣ Utilisation de GYM_LOGS — Calcul du besoin calorique

La table `gym_logs` contient le **métabolisme de base** (`metabolisme_de_base`) et la **dépense calorique théorique** (`depense_calorique_theorique`). On l'utilise pour estimer si l'utilisateur mange assez par rapport à son effort.

In [ ]:
# ---- Calcul de la balance énergétique depuis gym_logs ----
df_gym_clean = df_gym.copy()
df_gym_clean['metabolisme_de_base']       = pd.to_numeric(df_gym_clean['metabolisme_de_base'], errors='coerce')
df_gym_clean['depense_calorique_theorique']= pd.to_numeric(df_gym_clean['depense_calorique_theorique'], errors='coerce')
df_gym_clean['calories_burned']           = pd.to_numeric(df_gym_clean['calories_burned'], errors='coerce')

# Balance = dépense totale - métabolisme de base = calories brûlées par le sport seul
df_gym_clean['Balance_Energetique'] = df_gym_clean['depense_calorique_theorique'] - df_gym_clean['metabolisme_de_base']

print('=== Analyse du métabolisme par type d\'entraînement ===')
resume_gym = df_gym_clean.groupby('workout_type').agg(
    Metabolisme_Moyen  = ('metabolisme_de_base', 'mean'),
    Calories_Brulees   = ('calories_burned', 'mean'),
    Depense_Theorique  = ('depense_calorique_theorique', 'mean')
).round(0)
display(resume_gym)

metabolisme_moyen = df_gym_clean['metabolisme_de_base'].mean()
print(f'\nMétabolisme de base moyen (toutes séances) : {metabolisme_moyen:.0f} kcal/jour')

## 7️⃣ Utilisation du COMPENDIUM — Calcul de calories pour un sport inconnu

Si un utilisateur pratique un sport qui n'est pas dans ses logs, on cherche la **valeur MET** dans le compendium pour calculer ses calories brûlées.

In [ ]:
def calories_depuis_compendium(nom_sport, poids_kg, duree_heures):
    """
    Calcule les calories brûlées à partir du Compendium des activités.
    Formule : Calories = MET × Poids(kg) × Durée(h)
    """
    resultats = df_compendium[
        df_compendium['activity_description'].str.contains(nom_sport, case=False, na=False)
    ]
    
    if resultats.empty:
        return f"Sport '{nom_sport}' introuvable dans le Compendium."
    
    # On prend le MET moyen si plusieurs activités correspondent
    met_moyen = pd.to_numeric(resultats['met_value'], errors='coerce').mean()
    calories  = met_moyen * poids_kg * duree_heures
    
    print(f'Sport trouvé : {resultats.iloc[0]["activity_description"]}')
    print(f'Valeur MET   : {met_moyen:.1f}')
    print(f'Calories brûlées ({poids_kg}kg, {duree_heures}h) : {calories:.0f} kcal')
    return calories

# Test avec un exemple
print('=== Test : Natation, 70kg, 1h ===')
calories_depuis_compendium('swimming', 70, 1)
print()
print('=== Test : Yoga, 65kg, 0.5h ===')
calories_depuis_compendium('yoga', 65, 0.5)

## 8️⃣ Recommandation CIQUAL — Lier la prédiction IA à l'alimentation

On utilise la base CIQUAL pour recommander des aliments en fonction des besoins identifiés par l'IA.

In [ ]:
# ---- Nettoyage CIQUAL (valeurs numériques) ----
cols_num = ['calories_kcal_100g', 'proteines_g_100g', 'magnesium_mg_100g',
            'glucides_g_100g', 'lipides_g_100g', 'fibres_g_100g']
for col in cols_num:
    df_ciqual[col] = pd.to_numeric(df_ciqual[col], errors='coerce')

def recommander_aliments(besoin, top_n=5):
    """
    Recommande des aliments de la base CIQUAL selon un besoin nutritionnel.
    Besoins supportés : 'magnesium', 'proteines', 'glucides', 'fibres'
    """
    mapping = {
        'magnesium' : ('magnesium_mg_100g', 'Magnésium (mg/100g)'),
        'proteines' : ('proteines_g_100g',  'Protéines (g/100g)'),
        'glucides'  : ('glucides_g_100g',   'Glucides (g/100g)'),
        'fibres'    : ('fibres_g_100g',     'Fibres (g/100g)')
    }
    if besoin not in mapping:
        return f"Besoin '{besoin}' non reconnu. Choisir parmi : {list(mapping.keys())}"
    
    col, label = mapping[besoin]
    selection  = df_ciqual[['nom', 'sous_groupe', col]].dropna()
    # Exclure les valeurs aberrantes (algues séchées avec 2000mg de magnésium…)
    seuil = selection[col].quantile(0.98)
    selection = selection[selection[col] <= seuil]
    top = selection.nlargest(top_n, col)[['nom','sous_groupe', col]]
    top.columns = ['Aliment', 'Catégorie', label]
    return top

print('🥜 Top 5 aliments riches en MAGNÉSIUM (stress élevé) :')
display(recommander_aliments('magnesium'))

print('\n🍗 Top 5 aliments riches en PROTÉINES (récupération musculaire) :')
display(recommander_aliments('proteines'))

## 9️⃣ PIPELINE COMPLET IA-NAHA

Cette fonction est le **cœur d'IA-NAHA** : elle prend les données d'un utilisateur, active les 2 modèles IA, calcule les besoins nutritionnels, et retourne des recommandations issues de CIQUAL.

In [ ]:
def pipeline_IA_NAHA(age, bmi, genre, sport, intensite, stress, calories, pas,
                     bpm_moyen, bpm_repos, pression_sys, pression_dia,
                     hydratation, duree_minutes):
    """
    Pipeline complet IA-NAHA.
    Entrées  : données quotidiennes de l'utilisateur
    Sorties  : heures de sommeil recommandées + qualité prédite + conseils nutritionnels
    """
    print('=' * 60)
    print('       🤖 IA-NAHA — Analyse personnalisée')
    print('=' * 60)
    
    # ---- Encodage des variables texte ----
    genre_n    = 0 if genre.lower() in ['f','femme','female'] else 1
    sport_n    = hash(sport.lower()) % 20        # encodage simplifié
    intensite_n= {'low':0,'medium':1,'high':2}.get(intensite.lower(), 1)
    
    # ---- Variables calculées ----
    charge_cardiaque   = bpm_moyen - bpm_repos
    charge_activite    = duree_minutes * calories
    ratio_pas_calories = pas / (calories + 1)
    imc_stress         = bmi * stress
    
    # ---- MODÈLE 1 : Prédiction durée du sommeil ----
    entree_m1 = np.array([[age, bmi, genre_n, sport_n, intensite_n,
                           stress, calories, pas,
                           charge_cardiaque, charge_activite, ratio_pas_calories, imc_stress,
                           pression_sys, pression_dia, hydratation, duree_minutes]])
    
    heures_predites = modele_duree.predict(entree_m1)[0]
    
    # ---- MODÈLE 2 : Prédiction qualité du sommeil ----
    pal = min(int(duree_minutes / 60 * 10), 100)
    entree_m2   = np.array([[age, stress, pas, bpm_moyen, pal]])
    qualite_predite = modele_qualite.predict(entree_m2)[0]
    
    # ---- Interprétation ----
    etat_sommeil = '🟢 Bon' if qualite_predite >= 7 else ('🟡 Moyen' if qualite_predite >= 5 else '🔴 Mauvais')
    etat_stress  = '🔴 Élevé' if stress >= 7 else ('🟡 Modéré' if stress >= 4 else '🟢 Faible')
    
    print(f'\n📊 Profil : {age} ans | IMC={bmi:.1f} | Sport={sport}')
    print(f'   Charge cardiaque : {charge_cardiaque:.0f} BPM d\'écart')
    print(f'   Niveau de stress : {stress}/10 → {etat_stress}')
    print()
    print(f'😴 PRÉDICTION SOMMEIL')
    print(f'   Durée recommandée : {heures_predites:.1f}h')
    print(f'   Qualité estimée   : {qualite_predite}/10 → {etat_sommeil}')
    
    # ---- Recommandations nutritionnelles ----
    print()
    print('🥗 RECOMMANDATIONS NUTRITIONNELLES (Base CIQUAL)')
    
    if stress >= 6:
        print('   → Stress élevé détecté : Besoin en MAGNÉSIUM')
        print('     Aliments recommandés :')
        reco_mag = recommander_aliments('magnesium', top_n=3)
        for _, row in reco_mag.iterrows():
            print(f'       • {row["Aliment"]} ({row.iloc[2]:.0f} mg/100g)')
    
    if charge_activite > 10000:  # effort important
        print('   → Effort physique important : Besoin en PROTÉINES')
        reco_prot = recommander_aliments('proteines', top_n=3)
        for _, row in reco_prot.iterrows():
            print(f'       • {row["Aliment"]} ({row.iloc[2]:.0f} g/100g)')
    
    if heures_predites < 6:
        print('   → Risque de fatigue chronique : Augmentez les GLUCIDES complexes')
        reco_gluc = recommander_aliments('glucides', top_n=3)
        for _, row in reco_gluc.iterrows():
            print(f'       • {row["Aliment"]} ({row.iloc[2]:.0f} g/100g)')
    
    print('=' * 60)
    return heures_predites, qualite_predite


# ====== TEST DU PIPELINE ======
print('--- TEST 1 : Profil stressé, peu actif ---')
pipeline_IA_NAHA(
    age=32, bmi=25.4, genre='M', sport='Running', intensite='Medium',
    stress=8, calories=350, pas=4500,
    bpm_moyen=145, bpm_repos=72, pression_sys=125, pression_dia=80,
    hydratation=1.5, duree_minutes=30
)

print()
print('--- TEST 2 : Profil sportif, peu stressé ---')
pipeline_IA_NAHA(
    age=25, bmi=21.0, genre='F', sport='Cycling', intensite='High',
    stress=3, calories=650, pas=12000,
    bpm_moyen=160, bpm_repos=55, pression_sys=110, pression_dia=70,
    hydratation=2.5, duree_minutes=75
)

## 🔟 Bilan méthodologique

### Ce que nous avons fait

| Étape | Description | Tables utilisées |
|---|---|---|
| Modèle 1 | Régression pour prédire la **durée du sommeil** | `activite_globale`  |
| Modèle 2 | Classification pour prédire la **qualité du sommeil** | `sommeil_logs` |
| Calcul métabolique | Estimation des besoins caloriques | `gym_logs` |
| Calcul MET | Calories pour tout sport inconnu | `compendium_sports` |
| Recommandation | Aliments ciblés selon les besoins détectés | `ciqual_nutrition` |
| Profils | Contextualisation par âge/IMC/genre | `profil_utilisateur` |

### Interprétation du R²

Un R² entre 0.15 et 0.35 est **typique pour des données comportementales humaines**. Le sommeil dépend de nombreux facteurs non mesurables (caféine, lumière, stress non déclaré). L'IA capture néanmoins les tendances fortes comme l'impact du stress et de la charge cardiaque.

### Améliorations possibles
- Enrichir les données avec des mesures wearable (montre connectée)
- Ajouter des données alimentaires réelles pour fermer la boucle CIQUAL → sommeil
- Tester des modèles de Deep Learning (LSTM) sur des séries temporelles